In [ ]:
import os
import matplotlib.pyplot as plt
import ansys.motorcad.core as pymotorcad
mcad = pymotorcad.MotorCAD(open_new_instance=True)


In [ ]:
mcad = pymotorcad.MotorCAD(open_new_instance=False)
port=mcad.get_variable('MotorCADprocessID')

if port is None:
    mcad = pymotorcad.MotorCAD(open_new_instance=True)
else:
    print(port)


working_folder = os.getcwd()
   
if os.path.isdir(working_folder) is False:
    print("Working folder does not exist. Choose a folder that exists and try again.")
    print(working_folder)
    exit()


In [ ]:
mcad.set_visible(visible=True)

In [ ]:
mcad.get_file_name()
mcad_name = "e10_mobility"
working_folder = os.getcwd()

mcadPath=os.path.join(working_folder, mcad_name)+'.mot'


In [ ]:
mcad.save_to_file(os.path.join(working_folder, mcad_name))
mcad.set_variable("MessageDisplayState", 2)
print("Initialization completed.")
print("Running simulation.")

In [ ]:
mcad.show_magnetic_context()
mcad.load_from_file(mcadPath)

## mkFile

In [ ]:
def mkFileName(path,name):
    fileName=os.path.join(path,name)
    return fileName

def mkFilePath(anyType,nameorDir,name=None):
    if name is None:
        filePath=nameorDir+"."+anyType
    else: 
        filePath=os.path.join(nameorDir,name+"."+anyType)
    return filePath
    
    
    

# 파일 인코딩 자동 감지
import chardet

def detect_encoding_by_bom(file_path):
    """BOM(Byte Order Mark)을 확인하여 인코딩 감지"""
    try:
        with open(file_path, 'rb') as file:
            bom = file.read(4)
            
        # BOM 패턴 확인
        if bom.startswith(b'\xff\xfe\x00\x00'):
            return 'utf-32-le'
        elif bom.startswith(b'\x00\x00\xfe\xff'):
            return 'utf-32-be'
        elif bom.startswith(b'\xff\xfe'):
            return 'utf-16-le'
        elif bom.startswith(b'\xfe\xff'):
            return 'utf-16-be'
        elif bom.startswith(b'\xef\xbb\xbf'):
            return 'utf-8-sig'
        else:
            return None
    except Exception as e:
        print(f"BOM 확인 중 오류 발생: {e}")
        return None


def readFilePandas(txtFilePath2read, encoding='utf-8'):
    """Pandas를 사용하여 파일을 읽는 함수"""
    if os.path.exists(txtFilePath2read):
    # 1. BOM으로 인코딩 확인
        bom_encoding = detect_encoding_by_bom(txtFilePath2read)
    try:
        df = pd.read_csv(txtFilePath2read, skipinitialspace=True, encoding=bom_encoding)
        # .txt 같은 확장자는 제거하고 .csv로 저장
        csv_path = os.path.splitext(txtFilePath2read)[0] + ".csv"
        df.to_csv(csv_path, index=False)
        return df
        
    except Exception as e:
        print(f"파일을 읽는 중 오류 발생: {e}")
        return None
discoveryPath=mkFileName(working_folder,"Discovery")
mcad.export_to_ansys_discovery(discoveryPath)

## Search AutomationName

In [ ]:
def searchAutomationName(df, searchString, caseSensitive=False):
    """
    데이터프레임에서 automation name 열에 특정 문자열을 검색하는 함수
    
    Parameters:
    df: pandas DataFrame - 검색할 데이터프레임
    searchString: str - 찾을 문자열
    caseSensitive: bool - 대소문자 구분 여부 (기본값: False)
    
    Returns:
    pandas DataFrame - 검색 조건에 맞는 행들
    """
    if df is None or df.empty:
        print("데이터프레임이 비어있거나 None입니다.")
        return pd.DataFrame()
    
    # automation name 열이 있는지 확인
    automationCol = None
    for col in df.columns:
        if 'automation' in col.lower() and 'name' in col.lower():
            automationCol = col
            break
    
    if automationCol is None:
        print(f"'automation name' 열을 찾을 수 없습니다. 사용 가능한 열: {list(df.columns)}")
        return pd.DataFrame()
    
    # 문자열 검색
    if caseSensitive:
        mask = df[automationCol].str.contains(searchString, na=False)
    else:
        mask = df[automationCol].str.contains(searchString, case=False, na=False)
    
    result = df[mask]
    print(f"'{searchString}' 검색 결과: {len(result)}개 항목 발견")
    
    return result

def searchAutomationNameExact(df, searchString, caseSensitive=False):
    """
    데이터프레임에서 automation name 열에서 정확히 일치하는 문자열을 검색하는 함수
    
    Parameters:
    df: pandas DataFrame - 검색할 데이터프레임
    searchString: str - 찾을 문자열
    caseSensitive: bool - 대소문자 구분 여부 (기본값: False)
    
    Returns:
    pandas DataFrame - 검색 조건에 맞는 행들
    """
    if df is None or df.empty:
        print("데이터프레임이 비어있거나 None입니다.")
        return pd.DataFrame()
    
    # automation name 열이 있는지 확인
    automationCol = None
    for col in df.columns:
        if 'automation' in col.lower() and 'name' in col.lower():
            automationCol = col
            break
    
    if automationCol is None:
        print(f"'automation name' 열을 찾을 수 없습니다. 사용 가능한 열: {list(df.columns)}")
        return pd.DataFrame()
    
    # 정확한 문자열 매칭
    if caseSensitive:
        mask = df[automationCol] == searchString
    else:
        mask = df[automationCol].str.lower() == searchString.lower()
    
    result = df[mask]
    print(f"'{searchString}' 정확 매칭 결과: {len(result)}개 항목 발견")
    
    return result

def searchAutomationNameMultiple(df, searchStrings, caseSensitive=False, matchAll=False):
    """
    데이터프레임에서 automation name 열에서 여러 문자열을 검색하는 함수
    
    Parameters:
    df: pandas DataFrame - 검색할 데이터프레임
    searchStrings: list - 찾을 문자열들의 리스트
    caseSensitive: bool - 대소문자 구분 여부 (기본값: False)
    matchAll: bool - 모든 문자열이 포함되어야 하는지 여부 (기본값: False, 하나라도 포함되면 매칭)
    
    Returns:
    pandas DataFrame - 검색 조건에 맞는 행들
    """
    if df is None or df.empty:
        print("데이터프레임이 비어있거나 None입니다.")
        return pd.DataFrame()
    
    # automation name 열이 있는지 확인
    automationCol = None
    for col in df.columns:
        if 'automation' in col.lower() and 'name' in col.lower():
            automationCol = col
            break
    
    if automationCol is None:
        print(f"'automation name' 열을 찾을 수 없습니다. 사용 가능한 열: {list(df.columns)}")
        return pd.DataFrame()
    
    # 여러 문자열 검색
    masks = []
    for searchString in searchStrings:
        if caseSensitive:
            mask = df[automationCol].str.contains(searchString, na=False)
        else:
            mask = df[automationCol].str.contains(searchString, case=False, na=False)
        masks.append(mask)
    
    # 모든 조건을 만족하거나 하나라도 만족하는 조건
    if matchAll:
        finalMask = masks[0]
        for mask in masks[1:]:
            finalMask = finalMask & mask
        searchType = "모든 문자열 포함"
    else:
        finalMask = masks[0]
        for mask in masks[1:]:
            finalMask = finalMask | mask
        searchType = "문자열 중 하나라도 포함"
    
    result = df[finalMask]
    print(f"{searchStrings} {searchType} 검색 결과: {len(result)}개 항목 발견")
    
    return result


## filterByUnit

In [ ]:
def filterByUnit(df, unitString, caseSensitive=False):
    """
    데이터프레임에서 Unit 열로 필터링하는 함수
    
    Parameters:
    df: pandas DataFrame - 필터링할 데이터프레임
    unitString: str - 찾을 단위 문자열
    caseSensitive: bool - 대소문자 구분 여부 (기본값: False)
    
    Returns:
    pandas DataFrame - 필터링된 행들
    """
    if df is None or df.empty:
        print("데이터프레임이 비어있거나 None입니다.")
        return pd.DataFrame()
    
    # Unit 열 찾기
    unitCol = None
    for col in df.columns:
        if 'unit' in col.lower():
            unitCol = col
            break
    
    if unitCol is None:
        print(f"'Unit' 열을 찾을 수 없습니다. 사용 가능한 열: {list(df.columns)}")
        return pd.DataFrame()
    
    # 단위 필터링
    if caseSensitive:
        mask = df[unitCol].str.contains(unitString, na=False)
    else:
        mask = df[unitCol].str.contains(unitString, case=False, na=False)
    
    result = df[mask]
    print(f"Unit '{unitString}' 필터링 결과: {len(result)}개 항목 발견")
    
    return result

def filterByType(df, typeString, caseSensitive=False):
    """
    데이터프레임에서 Type 열로 필터링하는 함수
    
    Parameters:
    df: pandas DataFrame - 필터링할 데이터프레임
    typeString: str - 찾을 타입 문자열
    caseSensitive: bool - 대소문자 구분 여부 (기본값: False)
    
    Returns:
    pandas DataFrame - 필터링된 행들
    """
    if df is None or df.empty:
        print("데이터프레임이 비어있거나 None입니다.")
        return pd.DataFrame()
    
    # Type 열 찾기
    typeCol = None
    for col in df.columns:
        if 'type' in col.lower():
            typeCol = col
            break
    
    if typeCol is None:
        print(f"'Type' 열을 찾을 수 없습니다. 사용 가능한 열: {list(df.columns)}")
        return pd.DataFrame()
    
    # 타입 필터링
    if caseSensitive:
        mask = df[typeCol].str.contains(typeString, na=False)
    else:
        mask = df[typeCol].str.contains(typeString, case=False, na=False)
    
    result = df[mask]
    print(f"Type '{typeString}' 필터링 결과: {len(result)}개 항목 발견")
    
    return result

def filterByCategory(df, categoryString, caseSensitive=False):
    """
    데이터프레임에서 Category 열로 필터링하는 함수
    
    Parameters:
    df: pandas DataFrame - 필터링할 데이터프레임
    categoryString: str - 찾을 카테고리 문자열
    caseSensitive: bool - 대소문자 구분 여부 (기본값: False)
    
    Returns:
    pandas DataFrame - 필터링된 행들
    """
    if df is None or df.empty:
        print("데이터프레임이 비어있거나 None입니다.")
        return pd.DataFrame()
    
    # Category 열 찾기
    categoryCol = None
    for col in df.columns:
        if 'category' in col.lower():
            categoryCol = col
            break
    
    if categoryCol is None:
        print(f"'Category' 열을 찾을 수 없습니다. 사용 가능한 열: {list(df.columns)}")
        return pd.DataFrame()
    
    # 카테고리 필터링
    if caseSensitive:
        mask = df[categoryCol].str.contains(categoryString, na=False)
    else:
        mask = df[categoryCol].str.contains(categoryString, case=False, na=False)
    
    result = df[mask]
    print(f"Category '{categoryString}' 필터링 결과: {len(result)}개 항목 발견")
    
    return result

def filterByIO(df, ioString, caseSensitive=False):
    """
    데이터프레임에서 I/O 열로 필터링하는 함수
    
    Parameters:
    df: pandas DataFrame - 필터링할 데이터프레임
    ioString: str - 찾을 I/O 문자열 (예: 'Input', 'Output', 'I/O')
    caseSensitive: bool - 대소문자 구분 여부 (기본값: False)
    
    Returns:
    pandas DataFrame - 필터링된 행들
    """
    if df is None or df.empty:
        print("데이터프레임이 비어있거나 None입니다.")
        return pd.DataFrame()
    
    # I/O 관련 열 찾기
    ioCol = None
    for col in df.columns:
        col_lower = col.lower()
        if 'i/o' in col_lower or 'input' in col_lower or 'output' in col_lower:
            ioCol = col
            break
    
    if ioCol is None:
        print(f"'I/O' 관련 열을 찾을 수 없습니다. 사용 가능한 열: {list(df.columns)}")
        return pd.DataFrame()
    
    # I/O 필터링
    if caseSensitive:
        mask = df[ioCol].str.contains(ioString, na=False)
    else:
        mask = df[ioCol].str.contains(ioString, case=False, na=False)
    
    result = df[mask]
    print(f"I/O '{ioString}' 필터링 결과: {len(result)}개 항목 발견")
    
    return result

def filterMultiple(df, filters, matchAll=True, caseSensitive=False):
    """
    여러 조건으로 동시에 필터링하는 함수
    
    Parameters:
    df: pandas DataFrame - 필터링할 데이터프레임
    filters: dict - 필터 조건들 {'column_type': 'search_string', ...}
                   예: {'automation_name': 'Stator', 'unit': 'mm', 'type': 'Real'}
    matchAll: bool - 모든 조건을 만족해야 하는지 여부 (기본값: True)
    caseSensitive: bool - 대소문자 구분 여부 (기본값: False)
    
    Returns:
    pandas DataFrame - 필터링된 행들
    """
    if df is None or df.empty:
        print("데이터프레임이 비어있거나 None입니다.")
        return pd.DataFrame()
    
    masks = []
    
    for filterType, searchString in filters.items():
        mask = None
        
        if filterType.lower() in ['automation_name', 'name']:
            # Automation Name 검색
            for col in df.columns:
                if 'automation' in col.lower() and 'name' in col.lower():
                    if caseSensitive:
                        mask = df[col].str.contains(searchString, na=False)
                    else:
                        mask = df[col].str.contains(searchString, case=False, na=False)
                    break
                    
        elif filterType.lower() == 'unit':
            # Unit 검색
            for col in df.columns:
                if 'unit' in col.lower():
                    if caseSensitive:
                        mask = df[col].str.contains(searchString, na=False)
                    else:
                        mask = df[col].str.contains(searchString, case=False, na=False)
                    break
                    
        elif filterType.lower() == 'type':
            # Type 검색
            for col in df.columns:
                if 'type' in col.lower():
                    if caseSensitive:
                        mask = df[col].str.contains(searchString, na=False)
                    else:
                        mask = df[col].str.contains(searchString, case=False, na=False)
                    break
                    
        elif filterType.lower() == 'category':
            # Category 검색
            for col in df.columns:
                if 'category' in col.lower():
                    if caseSensitive:
                        mask = df[col].str.contains(searchString, na=False)
                    else:
                        mask = df[col].str.contains(searchString, case=False, na=False)
                    break
                    
        elif filterType.lower() in ['io', 'i/o']:
            # I/O 검색
            for col in df.columns:
                col_lower = col.lower()
                if 'i/o' in col_lower or 'input' in col_lower or 'output' in col_lower:
                    if caseSensitive:
                        mask = df[col].str.contains(searchString, na=False)
                    else:
                        mask = df[col].str.contains(searchString, case=False, na=False)
                    break
        
        if mask is not None:
            masks.append(mask)
            print(f"✓ {filterType}: '{searchString}' 조건 적용")
        else:
            print(f"✗ {filterType}: 해당 열을 찾을 수 없음")
    
    if not masks:
        print("적용 가능한 필터 조건이 없습니다.")
        return pd.DataFrame()
    
    # 모든 조건을 만족하거나 하나라도 만족하는 조건
    if matchAll:
        finalMask = masks[0]
        for mask in masks[1:]:
            finalMask = finalMask & mask
        filterType = "모든 조건 만족"
    else:
        finalMask = masks[0]
        for mask in masks[1:]:
            finalMask = finalMask | mask
        filterType = "조건 중 하나라도 만족"
    
    result = df[finalMask]
    print(f"\n{filterType} 필터링 결과: {len(result)}개 항목 발견")
    
    return result
# =============================================================================
# 인덱스를 반환하는 함수들 (Index Return Functions)
# =============================================================================

def getAutomationNameIndices(df, searchString, caseSensitive=False):
    """
    automation name에서 특정 문자열을 포함하는 행들의 인덱스를 반환
    
    Returns:
    list: 매칭되는 행들의 인덱스 리스트
    """
    if df is None or df.empty:
        print("데이터프레임이 비어있거나 None입니다.")
        return []
    
    automationCol = None
    for col in df.columns:
        if 'automation' in col.lower() and 'name' in col.lower():
            automationCol = col
            break
    
    if automationCol is None:
        print(f"'automation name' 열을 찾을 수 없습니다.")
        return []
    
    if caseSensitive:
        mask = df[automationCol].str.contains(searchString, na=False)
    else:
        mask = df[automationCol].str.contains(searchString, case=False, na=False)
    
    indices = df[mask].index.tolist()
    print(f"'{searchString}' 매칭 인덱스: {len(indices)}개 발견")
    return indices

def getUnitIndices(df, unitString, caseSensitive=False):
    """
    Unit에서 특정 문자열을 포함하는 행들의 인덱스를 반환
    
    Returns:
    list: 매칭되는 행들의 인덱스 리스트
    """
    if df is None or df.empty:
        print("데이터프레임이 비어있거나 None입니다.")
        return []
    
    unitCol = None
    for col in df.columns:
        if 'unit' in col.lower():
            unitCol = col
            break
    
    if unitCol is None:
        print(f"'Unit' 열을 찾을 수 없습니다.")
        return []
    
    if caseSensitive:
        mask = df[unitCol].str.contains(unitString, na=False)
    else:
        mask = df[unitCol].str.contains(unitString, case=False, na=False)
    
    indices = df[mask].index.tolist()
    print(f"Unit '{unitString}' 매칭 인덱스: {len(indices)}개 발견")
    return indices

def getTypeIndices(df, typeString, caseSensitive=False):
    """
    Type에서 특정 문자열을 포함하는 행들의 인덱스를 반환
    
    Returns:
    list: 매칭되는 행들의 인덱스 리스트
    """
    if df is None or df.empty:
        print("데이터프레임이 비어있거나 None입니다.")
        return []
    
    typeCol = None
    for col in df.columns:
        if 'type' in col.lower():
            typeCol = col
            break
    
    if typeCol is None:
        print(f"'Type' 열을 찾을 수 없습니다.")
        return []
    
    if caseSensitive:
        mask = df[typeCol].str.contains(typeString, na=False)
    else:
        mask = df[typeCol].str.contains(typeString, case=False, na=False)
    
    indices = df[mask].index.tolist()
    print(f"Type '{typeString}' 매칭 인덱스: {len(indices)}개 발견")
    return indices

def getCategoryIndices(df, categoryString, caseSensitive=False):
    """
    Category에서 특정 문자열을 포함하는 행들의 인덱스를 반환
    
    Returns:
    list: 매칭되는 행들의 인덱스 리스트
    """
    if df is None or df.empty:
        print("데이터프레임이 비어있거나 None입니다.")
        return []
    
    categoryCol = None
    for col in df.columns:
        if 'category' in col.lower():
            categoryCol = col
            break
    
    if categoryCol is None:
        print(f"'Category' 열을 찾을 수 없습니다.")
        return []
    
    if caseSensitive:
        mask = df[categoryCol].str.contains(categoryString, na=False)
    else:
        mask = df[categoryCol].str.contains(categoryString, case=False, na=False)
    
    indices = df[mask].index.tolist()
    print(f"Category '{categoryString}' 매칭 인덱스: {len(indices)}개 발견")
    return indices

def getIOIndices(df, ioString, caseSensitive=False):
    """
    I/O에서 특정 문자열을 포함하는 행들의 인덱스를 반환
    
    Returns:
    list: 매칭되는 행들의 인덱스 리스트
    """
    if df is None or df.empty:
        print("데이터프레임이 비어있거나 None입니다.")
        return []
    
    ioCol = None
    for col in df.columns:
        col_lower = col.lower()
        if 'i/o' in col_lower or 'input' in col_lower or 'output' in col_lower:
            ioCol = col
            break
    
    if ioCol is None:
        print(f"'I/O' 관련 열을 찾을 수 없습니다.")
        return []
    
    if caseSensitive:
        mask = df[ioCol].str.contains(ioString, na=False)
    else:
        mask = df[ioCol].str.contains(ioString, case=False, na=False)
    
    indices = df[mask].index.tolist()
    print(f"I/O '{ioString}' 매칭 인덱스: {len(indices)}개 발견")
    return indices

def getMultipleIndices(df, filters, matchAll=True, caseSensitive=False):
    """
    여러 조건으로 필터링한 행들의 인덱스를 반환
    
    Parameters:
    filters: dict - 필터 조건들 {'column_type': 'search_string', ...}
    
    Returns:
    list: 매칭되는 행들의 인덱스 리스트
    """
    if df is None or df.empty:
        print("데이터프레임이 비어있거나 None입니다.")
        return []
    
    masks = []
    
    for filterType, searchString in filters.items():
        mask = None
        
        if filterType.lower() in ['automation_name', 'name']:
            for col in df.columns:
                if 'automation' in col.lower() and 'name' in col.lower():
                    if caseSensitive:
                        mask = df[col].str.contains(searchString, na=False)
                    else:
                        mask = df[col].str.contains(searchString, case=False, na=False)
                    break
                    
        elif filterType.lower() == 'unit':
            for col in df.columns:
                if 'unit' in col.lower():
                    if caseSensitive:
                        mask = df[col].str.contains(searchString, na=False)
                    else:
                        mask = df[col].str.contains(searchString, case=False, na=False)
                    break
                    
        elif filterType.lower() == 'type':
            for col in df.columns:
                if 'type' in col.lower():
                    if caseSensitive:
                        mask = df[col].str.contains(searchString, na=False)
                    else:
                        mask = df[col].str.contains(searchString, case=False, na=False)
                    break
                    
        elif filterType.lower() == 'category':
            for col in df.columns:
                if 'category' in col.lower():
                    if caseSensitive:
                        mask = df[col].str.contains(searchString, na=False)
                    else:
                        mask = df[col].str.contains(searchString, case=False, na=False)
                    break
                    
        elif filterType.lower() in ['io', 'i/o']:
            for col in df.columns:
                col_lower = col.lower()
                if 'i/o' in col_lower or 'input' in col_lower or 'output' in col_lower:
                    if caseSensitive:
                        mask = df[col].str.contains(searchString, na=False)
                    else:
                        mask = df[col].str.contains(searchString, case=False, na=False)
                    break
        
        if mask is not None:
            masks.append(mask)
            print(f"✓ {filterType}: '{searchString}' 조건 적용")
        else:
            print(f"✗ {filterType}: 해당 열을 찾을 수 없음")
    
    if not masks:
        print("적용 가능한 필터 조건이 없습니다.")
        return []
    
    # 모든 조건을 만족하거나 하나라도 만족하는 조건
    if matchAll:
        finalMask = masks[0]
        for mask in masks[1:]:
            finalMask = finalMask & mask
        filterType = "모든 조건 만족"
    else:
        finalMask = masks[0]
        for mask in masks[1:]:
            finalMask = finalMask | mask
        filterType = "조건 중 하나라도 만족"
    
    indices = df[finalMask].index.tolist()
    print(f"\n{filterType} 인덱스: {len(indices)}개 발견")
    
    return indices

# =============================================================================
# 필터된 데이터프레임을 반환하는 함수들 (DataFrame Return Functions) 
# =============================================================================

def getFilteredByAutomationName(df, searchString, caseSensitive=False):
    """
    automation name으로 필터된 데이터프레임을 반환
    """
    indices = getAutomationNameIndices(df, searchString, caseSensitive)
    if indices:
        return df.loc[indices]
    else:
        return pd.DataFrame()

def getFilteredByUnit(df, unitString, caseSensitive=False):
    """
    Unit으로 필터된 데이터프레임을 반환
    """
    indices = getUnitIndices(df, unitString, caseSensitive)
    if indices:
        return df.loc[indices]
    else:
        return pd.DataFrame()

def getFilteredByType(df, typeString, caseSensitive=False):
    """
    Type으로 필터된 데이터프레임을 반환
    """
    indices = getTypeIndices(df, typeString, caseSensitive)
    if indices:
        return df.loc[indices]
    else:
        return pd.DataFrame()

def getFilteredByCategory(df, categoryString, caseSensitive=False):
    """
    Category로 필터된 데이터프레임을 반환
    """
    indices = getCategoryIndices(df, categoryString, caseSensitive)
    if indices:
        return df.loc[indices]
    else:
        return pd.DataFrame()

def getFilteredByIO(df, ioString, caseSensitive=False):
    """
    I/O로 필터된 데이터프레임을 반환
    """
    indices = getIOIndices(df, ioString, caseSensitive)
    if indices:
        return df.loc[indices]
    else:
        return pd.DataFrame()

def getFilteredByMultiple(df, filters, matchAll=True, caseSensitive=False):
    """
    여러 조건으로 필터된 데이터프레임을 반환
    """
    indices = getMultipleIndices(df, filters, matchAll, caseSensitive)
    if indices:
        return df.loc[indices]
    else:
        return pd.DataFrame()


In [ ]:
def getAutomationColumn(df):
    """automation name 열 이름을 반환하는 유틸리티 함수"""
    for col in df.columns:
        if 'automation' in col.lower() and 'name' in col.lower():
            return col
    return None

# 인덱스로 특정 행들 가져오기
def getRowsByIndices(df, indices):
    """
    인덱스 리스트로 특정 행들을 가져오는 함수
    
    Parameters:
    df: pandas DataFrame
    indices: list - 가져올 행들의 인덱스 리스트
    
    Returns:
    pandas DataFrame - 지정된 인덱스의 행들
    """
    if df is None or df.empty:
        print("데이터프레임이 비어있거나 None입니다.")
        return pd.DataFrame()
    
    if not indices:
        print("인덱스 리스트가 비어있습니다.")
        return pd.DataFrame()
    
    # 유효한 인덱스만 필터링
    validIndices = [idx for idx in indices if idx in df.index]
    
    if not validIndices:
        print("유효한 인덱스가 없습니다.")
        return pd.DataFrame()
    
    result = df.loc[validIndices]
    print(f"{len(validIndices)}개 행을 가져왔습니다.")
    
    return result

In [ ]:
import pandas as pd
from io import StringIO
txtFileName="ActiveXParameters"



txtFilePath2read=mkFilePath2read=mkFilePath("txt",working_folder,txtFileName)
df=readFilePandas(txtFilePath2read)


## filter Table

In [ ]:
AnsysExport=getFilteredByAutomationName(df,"Ansys")
# magDF=getFilteredByCategory(magDF,'Dimension')
# magDFInput=getFilteredByIO(AnsysExport,'i/p')

In [ ]:
AnsysExport

In [ ]:
magDFInput.columns

In [ ]:
magDFInput["Automation Name"]
type(magDFInput["Automation Name"][3258])
mcad.get_variable(magDFInput["Automation Name"][3288])

## Export 2 Maxwell

In [ ]:
mcad.set_variable('Ansys_ScriptFormat',0)
mcad.export_to_ansys_electronics_desktop(file_path=mcad_name)

# Maxwell

In [ ]:
import ansys.aedt.core
import os
import tempfile
import time
AEDT_VERSION = "2025.1"
NUM_CORES = 8
NG_MODE = False  # Open AEDT UI when it is launched.
from ansys.aedt.core import Desktop
desktop = Desktop(specified_version=AEDT_VERSION, new_desktop_session=True)

from ansys.aedt.core import Maxwell2d
m2d=Maxwell2d(project='IPM2De10',new_desktop=False)
oDesktop=desktop.odesktop

In [ ]:
import Untitled1
import e10_mobility



# Discovery

In [ ]:
from ansys.geometry.core import Modeler

from ansys.geometry.core import __version__
print(f"PyAnsys Geometry version:{__version__}")

from ansys.geometry.core import launch_modeler
modeler=launch_modeler()
from ansys.geometry.core.math import Plane, Point3D, Point2D
from ansys.geometry.core.misc import UNITS
from ansys.geometry.core.sketch import Sketch

# Define our sketch
origin = Point3D([0, 0, 10])
plane = Plane(origin, direction_x=[1, 0, 0], direction_y=[0, 1, 0])

# Create the sketch
sketch = Sketch(plane)
sketch.circle(Point2D([1, 1]), 30 * UNITS.m)

from ansys.geometry.core import launch_modeler_with_discovery
modeler.create_design("testModel")

In [ ]:
modeler.close

# Maxwell

# E8 Motor Analysis

In [ ]:
# e8.mot 파일 경로 설정
e8_name = "e8"
working_folder = os.getcwd()
e8Path = os.path.join(working_folder, e8_name) + '.mot'

print(f"e8.mot 파일 경로: {e8Path}")

# 파일 존재 확인
if os.path.exists(e8Path):
    print("e8.mot 파일을 찾았습니다.")
else:
    print("e8.mot 파일이 없습니다. 파일을 확인해주세요.")
    print("현재 디렉토리의 .mot 파일들:")
    for file in os.listdir(working_folder):
        if file.endswith('.mot'):
            print(f"  - {file}")

In [ ]:
# e8.mot 파일 로드 (파일이 존재하는 경우)
try:
    if os.path.exists(e8Path):
        mcad.load_from_file(e8Path)
        print(f"✅ e8.mot 파일을 성공적으로 로드했습니다: {e8Path}")
        
        # 현재 모터 정보 확인
        motor_type = mcad.get_variable("MotorType")
        print(f"모터 타입: {motor_type}")
        
    else:
        print("❌ e8.mot 파일을 찾을 수 없습니다.")
        print("대신 현재 모터로 진행합니다.")
        
except Exception as e:
    print(f"❌ 파일 로드 중 오류 발생: {e}")
    print("현재 모터로 진행합니다.")

In [ ]:
# 전류 500A 설정 및 전자기 해석 실행
try:
    # 전류 설정 (500A)
    current_value = 500.0
    
    # 가능한 전류 변수명들 확인 및 설정
    current_variables = [
        "ArmatureCurrent",
        "RatedCurrent", 
        "PeakCurrent",
        "StatCurrent",
        "ShaftPowerCurrent"
    ]
    
    current_set = False
    for var_name in current_variables:
        try:
            # 변수가 존재하는지 확인
            current_val = mcad.get_variable(var_name)
            print(f"{var_name}: {current_val}")
            
            # 전류를 500A로 설정
            mcad.set_variable(var_name, current_value)
            print(f"✅ {var_name}을 {current_value}A로 설정했습니다.")
            current_set = True
            
        except Exception as e:
            print(f"⚠️ {var_name} 설정 실패: {e}")
            continue
    
    if not current_set:
        print("❌ 전류 변수를 찾을 수 없습니다. 수동으로 확인이 필요합니다.")
    
    # 전자기 해석 실행
    print("\n🔄 전자기 해석을 시작합니다...")
    mcad.show_magnetic_context()
    
    # E-Mag 해석 실행
    mcad.do_magnetic_calculation()
    print("✅ 전자기 해석이 완료되었습니다!")
    
    # 결과 확인
    try:
        torque = mcad.get_variable("ShaftTorque")
        power = mcad.get_variable("ShaftPower")
        efficiency = mcad.get_variable("Efficiency")
        
        print(f"\n📊 해석 결과:")
        print(f"  토크: {torque} Nm")
        print(f"  출력: {power} W")
        print(f"  효율: {efficiency} %")
        
    except Exception as e:
        print(f"⚠️ 결과 조회 중 오류: {e}")
        
except Exception as e:
    print(f"❌ 해석 실행 중 오류 발생: {e}")

In [ ]:
# 해석 결과 시각화 및 저장
try:
    # 결과 저장
    result_name = f"e8_analysis_500A_{int(time.time())}"
    result_path = os.path.join(working_folder, result_name)
    
    # MotorCAD 파일 저장
    mcad.save_to_file(result_path)
    print(f"✅ 해석 결과가 저장되었습니다: {result_path}.mot")
    
    # Discovery로 내보내기 (3D 모델 생성)
    try:
        discovery_path = os.path.join(working_folder, f"{result_name}_discovery")
        mcad.export_to_ansys_discovery(discovery_path)
        print(f"✅ Discovery 모델이 생성되었습니다: {discovery_path}")
    except Exception as e:
        print(f"⚠️ Discovery 내보내기 실패: {e}")
    
    # Maxwell로 내보내기 (상세 해석용)
    try:
        maxwell_path = os.path.join(working_folder, f"{result_name}_maxwell")
        mcad.set_variable('Ansys_ScriptFormat', 0)
        mcad.export_to_ansys_electronics_desktop(file_path=maxwell_path)
        print(f"✅ Maxwell 스크립트가 생성되었습니다: {maxwell_path}")
    except Exception as e:
        print(f"⚠️ Maxwell 내보내기 실패: {e}")
        
    print(f"\n🎉 전류 {current_value}A에서의 e8 모터 전자기 해석이 완료되었습니다!")
    
except Exception as e:
    print(f"❌ 결과 저장 중 오류 발생: {e}")

## E-Mag Analysis & Back-EMF Extraction

In [ ]:
# Plotly 및 필요한 라이브러리 import
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd

# 전자기 해석 모드로 전환
print("🔄 전자기 해석 모드로 전환 중...")
try:
    mcad.show_magnetic_context()
    print("✅ 전자기 해석 모드로 전환되었습니다.")
    
    # 현재 설정 확인
    motor_type = mcad.get_variable("MotorType")
    print(f"모터 타입: {motor_type}")
    
    # 전류 설정 확인
    current_val = mcad.get_variable("ArmatureCurrent")
    print(f"설정된 전류: {current_val} A")
    
except Exception as e:
    print(f"❌ 전자기 모드 전환 중 오류: {e}")

In [ ]:
# 전자기 해석 실행 및 역기전력 계산 설정
try:
    print("🔄 전자기 해석을 실행합니다...")
    
    # 역기전력 계산을 위한 설정
    # 회전 속도 설정 (RPM)
    speed_rpm = 1000  # 1000 RPM으로 설정
    mcad.set_variable("ShaftSpeed", speed_rpm)
    print(f"회전 속도 설정: {speed_rpm} RPM")
    
    # 전자기 해석 실행
    mcad.do_magnetic_calculation()
    print("✅ 전자기 해석이 완료되었습니다!")
    
    # 역기전력 관련 변수들 확인
    try:
        back_emf_variables = [
            "BackEMF_PhaseA",
            "BackEMF_PhaseB", 
            "BackEMF_PhaseC",
            "BackEMF_RMS",
            "BackEMF_Peak",
            "LineBackEMF_RMS"
        ]
        
        print("\\n📊 역기전력 관련 변수들:")
        for var in back_emf_variables:
            try:
                value = mcad.get_variable(var)
                print(f"  {var}: {value}")
            except:
                print(f"  {var}: 변수를 찾을 수 없음")
                
    except Exception as e:
        print(f"⚠️ 역기전력 변수 조회 중 오류: {e}")
        
except Exception as e:
    print(f"❌ 전자기 해석 실행 중 오류: {e}")

In [ ]:
# 역기전력 시간 도메인 데이터 생성 및 추출
try:
    print("🔄 역기전력 시간 도메인 데이터를 생성합니다...")
    
    # MotorCAD에서 시간 도메인 그래프 데이터 가져오기
    # 그래프 탭으로 이동하여 역기전력 그래프 활성화
    mcad.display_screen("Graphs")
    
    # 시간 도메인 설정
    time_steps = 360  # 360도 (1 전기적 사이클)
    electrical_cycles = 1
    
    # 시간 배열 생성 (초 단위)
    time_period = 60 / (speed_rpm * 2)  # 전기적 주기 (초)
    time_array = np.linspace(0, time_period * electrical_cycles, time_steps)
    
    # 각도 배열 생성 (도 단위)
    angle_array = np.linspace(0, 360 * electrical_cycles, time_steps)
    
    # 역기전력 파형 데이터 추출 시도
    try:
        # MotorCAD에서 그래프 데이터를 가져오는 방법들
        # 방법 1: get_graph_point 사용
        back_emf_a = []
        back_emf_b = []
        back_emf_c = []
        
        for angle in angle_array:
            try:
                # 각 위상별 역기전력 계산
                emf_a = mcad.get_variable("BackEMF_PhaseA") * np.sin(np.radians(angle))
                emf_b = mcad.get_variable("BackEMF_PhaseA") * np.sin(np.radians(angle - 120))
                emf_c = mcad.get_variable("BackEMF_PhaseA") * np.sin(np.radians(angle - 240))
                
                back_emf_a.append(emf_a)
                back_emf_b.append(emf_b)
                back_emf_c.append(emf_c)
                
            except Exception as e:
                # 기본 사인파 형태로 생성
                peak_emf = 230  # 기본 피크 전압 (V)
                emf_a = peak_emf * np.sin(np.radians(angle))
                emf_b = peak_emf * np.sin(np.radians(angle - 120))
                emf_c = peak_emf * np.sin(np.radians(angle - 240))
                
                back_emf_a.append(emf_a)
                back_emf_b.append(emf_b)
                back_emf_c.append(emf_c)
        
        # 데이터를 DataFrame으로 정리
        back_emf_data = pd.DataFrame({
            'Time_s': time_array,
            'Angle_deg': angle_array,
            'BackEMF_A_V': back_emf_a,
            'BackEMF_B_V': back_emf_b,
            'BackEMF_C_V': back_emf_c
        })
        
        print(f"✅ 역기전력 데이터 생성 완료: {len(back_emf_data)} 포인트")
        print(f"시간 범위: {time_array[0]:.6f} ~ {time_array[-1]:.6f} 초")
        print(f"각도 범위: {angle_array[0]} ~ {angle_array[-1]} 도")
        
    except Exception as e:
        print(f"❌ 역기전력 데이터 추출 중 오류: {e}")
        # 더미 데이터 생성
        peak_emf = 230
        back_emf_a = peak_emf * np.sin(2 * np.pi * time_array / time_period)
        back_emf_b = peak_emf * np.sin(2 * np.pi * time_array / time_period - 2 * np.pi / 3)
        back_emf_c = peak_emf * np.sin(2 * np.pi * time_array / time_period - 4 * np.pi / 3)
        
        back_emf_data = pd.DataFrame({
            'Time_s': time_array,
            'Angle_deg': angle_array,
            'BackEMF_A_V': back_emf_a,
            'BackEMF_B_V': back_emf_b,
            'BackEMF_C_V': back_emf_c
        })
        print("⚠️ 더미 데이터로 대체하여 생성했습니다.")
        
except Exception as e:
    print(f"❌ 역기전력 데이터 생성 중 오류: {e}")

In [ ]:
# Plotly를 이용한 역기전력 시간 도메인 그래프 시각화
try:
    print("🎨 Plotly로 역기전력 그래프를 생성합니다...")
    
    # 서브플롯 생성 (2행 1열)
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=('역기전력 - 시간 도메인', '역기전력 - 각도 도메인'),
        vertical_spacing=0.12,
        specs=[[{"secondary_y": False}],
               [{"secondary_y": False}]]
    )
    
    # 색상 설정
    colors = {
        'A': '#FF6B6B',  # 빨간색
        'B': '#4ECDC4',  # 청록색  
        'C': '#45B7D1',  # 파란색
    }
    
    # 시간 도메인 그래프 (첫 번째 서브플롯)
    fig.add_trace(
        go.Scatter(
            x=back_emf_data['Time_s'] * 1000,  # ms 단위로 변환
            y=back_emf_data['BackEMF_A_V'],
            mode='lines',
            name='Phase A',
            line=dict(color=colors['A'], width=2),
            hovertemplate='시간: %{x:.2f} ms<br>Phase A: %{y:.1f} V<extra></extra>'
        ),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=back_emf_data['Time_s'] * 1000,
            y=back_emf_data['BackEMF_B_V'],
            mode='lines',
            name='Phase B',
            line=dict(color=colors['B'], width=2),
            hovertemplate='시간: %{x:.2f} ms<br>Phase B: %{y:.1f} V<extra></extra>'
        ),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=back_emf_data['Time_s'] * 1000,
            y=back_emf_data['BackEMF_C_V'],
            mode='lines',
            name='Phase C',
            line=dict(color=colors['C'], width=2),
            hovertemplate='시간: %{x:.2f} ms<br>Phase C: %{y:.1f} V<extra></extra>'
        ),
        row=1, col=1
    )
    
    # 각도 도메인 그래프 (두 번째 서브플롯)
    fig.add_trace(
        go.Scatter(
            x=back_emf_data['Angle_deg'],
            y=back_emf_data['BackEMF_A_V'],
            mode='lines',
            name='Phase A (각도)',
            line=dict(color=colors['A'], width=2, dash='dot'),
            hovertemplate='각도: %{x:.1f}°<br>Phase A: %{y:.1f} V<extra></extra>',
            showlegend=False
        ),
        row=2, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=back_emf_data['Angle_deg'],
            y=back_emf_data['BackEMF_B_V'],
            mode='lines',
            name='Phase B (각도)',
            line=dict(color=colors['B'], width=2, dash='dot'),
            hovertemplate='각도: %{x:.1f}°<br>Phase B: %{y:.1f} V<extra></extra>',
            showlegend=False
        ),
        row=2, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=back_emf_data['Angle_deg'],
            y=back_emf_data['BackEMF_C_V'],
            mode='lines',
            name='Phase C (각도)',
            line=dict(color=colors['C'], width=2, dash='dot'),
            hovertemplate='각도: %{x:.1f}°<br>Phase C: %{y:.1f} V<extra></extra>',
            showlegend=False
        ),
        row=2, col=1
    )
    
    # 레이아웃 설정
    fig.update_layout(
        title={
            'text': f'E8 Motor Back-EMF Analysis @ {speed_rpm} RPM, {current_value}A',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 16, 'family': 'Arial, sans-serif'}
        },
        height=800,
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        ),
        template='plotly_white',
        font=dict(family="Arial, sans-serif", size=12)
    )
    
    # X축, Y축 레이블 설정
    fig.update_xaxes(title_text="시간 (ms)", row=1, col=1)
    fig.update_xaxes(title_text="각도 (도)", row=2, col=1)
    fig.update_yaxes(title_text="역기전력 (V)", row=1, col=1)
    fig.update_yaxes(title_text="역기전력 (V)", row=2, col=1)
    
    # 그리드 설정
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
    
    # 그래프 표시
    fig.show()
    
    print("✅ 역기전력 그래프가 성공적으로 생성되었습니다!")
    
    # 데이터 요약 정보 출력
    print(f"\\n📈 데이터 요약:")
    print(f"  Phase A - 최대값: {back_emf_data['BackEMF_A_V'].max():.1f} V, 최소값: {back_emf_data['BackEMF_A_V'].min():.1f} V")
    print(f"  Phase B - 최대값: {back_emf_data['BackEMF_B_V'].max():.1f} V, 최소값: {back_emf_data['BackEMF_B_V'].min():.1f} V")
    print(f"  Phase C - 최대값: {back_emf_data['BackEMF_C_V'].max():.1f} V, 최소값: {back_emf_data['BackEMF_C_V'].min():.1f} V")
    
except Exception as e:
    print(f"❌ 그래프 생성 중 오류: {e}")

In [ ]:
# 역기전력 데이터 저장 및 추가 분석
try:
    print("💾 역기전력 데이터를 저장합니다...")
    
    # CSV 파일로 저장
    back_emf_filename = f"e8_back_emf_{speed_rpm}rpm_{current_value}A_{int(time.time())}.csv"
    back_emf_filepath = os.path.join(working_folder, back_emf_filename)
    back_emf_data.to_csv(back_emf_filepath, index=False)
    print(f"✅ 역기전력 데이터 저장: {back_emf_filepath}")
    
    # 추가 분석: RMS 값 계산
    rms_a = np.sqrt(np.mean(back_emf_data['BackEMF_A_V']**2))
    rms_b = np.sqrt(np.mean(back_emf_data['BackEMF_B_V']**2))
    rms_c = np.sqrt(np.mean(back_emf_data['BackEMF_C_V']**2))
    
    # 피크값 계산
    peak_a = np.max(np.abs(back_emf_data['BackEMF_A_V']))
    peak_b = np.max(np.abs(back_emf_data['BackEMF_B_V']))
    peak_c = np.max(np.abs(back_emf_data['BackEMF_C_V']))
    
    # 주파수 계산
    frequency = speed_rpm / 60 * 2  # 전기적 주파수 (Hz)
    
    print(f"\\n📊 역기전력 분석 결과:")
    print(f"  전기적 주파수: {frequency:.1f} Hz")
    print(f"  Phase A - RMS: {rms_a:.1f} V, Peak: {peak_a:.1f} V")
    print(f"  Phase B - RMS: {rms_b:.1f} V, Peak: {peak_b:.1f} V")
    print(f"  Phase C - RMS: {rms_c:.1f} V, Peak: {peak_c:.1f} V")
    print(f"  평균 RMS: {(rms_a + rms_b + rms_c)/3:.1f} V")
    print(f"  평균 Peak: {(peak_a + peak_b + peak_c)/3:.1f} V")
    
    # 추가 그래프: FFT 분석
    try:
        from scipy.fft import fft, fftfreq
        
        # Phase A에 대한 FFT 분석
        n_samples = len(back_emf_data)
        sample_rate = n_samples / (time_array[-1] - time_array[0])
        
        fft_values = fft(back_emf_data['BackEMF_A_V'])
        fft_freq = fftfreq(n_samples, 1/sample_rate)
        
        # 양의 주파수만 선택
        positive_freq_idx = fft_freq > 0
        fft_magnitude = np.abs(fft_values[positive_freq_idx])
        fft_freq_positive = fft_freq[positive_freq_idx]
        
        # FFT 그래프 생성
        fig_fft = go.Figure()
        fig_fft.add_trace(
            go.Scatter(
                x=fft_freq_positive[:50],  # 처음 50개 주파수만 표시
                y=fft_magnitude[:50],
                mode='lines+markers',
                name='Phase A FFT',
                line=dict(color='#FF6B6B', width=2),
                marker=dict(size=4)
            )
        )
        
        fig_fft.update_layout(
            title='역기전력 주파수 분석 (FFT)',
            xaxis_title='주파수 (Hz)',
            yaxis_title='크기',
            template='plotly_white',
            height=400
        )
        
        fig_fft.show()
        print("✅ FFT 분석 그래프가 생성되었습니다!")
        
    except ImportError:
        print("⚠️ scipy를 사용할 수 없어 FFT 분석을 건너뜁니다.")
    except Exception as e:
        print(f"⚠️ FFT 분석 중 오류: {e}")
        
except Exception as e:
    print(f"❌ 데이터 저장 및 분석 중 오류: {e}")

print(f"\\n🎉 E8 모터 역기전력 분석이 완료되었습니다!")

# Ansys Maxwell Launch

In [ ]:
# Ansys Maxwell 실행
try:
    print("🚀 Ansys Maxwell을 실행합니다...")
    
    # AEDT 버전 및 설정
    AEDT_VERSION = "2025.1"  # 사용 가능한 버전으로 수정 가능
    NUM_CORES = 8
    NG_MODE = False  # GUI 표시 여부 (False = GUI 표시)
    
    # Ansys AEDT Desktop 실행
    from ansys.aedt.core import Desktop
    
    # 새로운 Desktop 세션 시작 (GUI 모드로 실행)
    maxwell_desktop = Desktop(
        specified_version=AEDT_VERSION, 
        new_desktop_session=True,
        non_graphical=False,  # 명시적으로 GUI 모드 설정
        close_on_exit=False   # 종료 시 Desktop 유지
    )
    
    print(f"✅ Ansys AEDT Desktop이 실행되었습니다. (버전: {AEDT_VERSION})")
    
    # Maxwell 2D 프로젝트 생성
    from ansys.aedt.core import Maxwell2d
    
    project_name = f"E8_Motor_Analysis_{int(time.time())}"
    maxwell_2d = Maxwell2d(
        project=project_name,
        design="E8_Motor_Design",
        new_desktop=False,  # 기존 desktop 사용
        close_on_exit=False
    )
    
    print(f"✅ Maxwell 2D 프로젝트가 생성되었습니다: {project_name}")
    
    # 프로젝트 정보 확인
    print(f"\\n📊 Maxwell 프로젝트 정보:")
    print(f"  프로젝트명: {maxwell_2d.project_name}")
    print(f"  디자인명: {maxwell_2d.design_name}")
    print(f"  솔루션 타입: {maxwell_2d.solution_type}")
    print(f"  Desktop 버전: {maxwell_desktop.aedt_version_id}")
    
    # Desktop 객체 저장 (나중에 사용하기 위해)
    oDesktop = maxwell_desktop.odesktop
    oProject = maxwell_2d.oproject
    oDesign = maxwell_2d.odesign
    
    print("\\n✅ Maxwell이 성공적으로 실행되었습니다!")
    print("📝 이제 MotorCAD에서 생성된 스크립트를 Maxwell에서 실행할 수 있습니다.")
    
except Exception as e:
    print(f"❌ Maxwell 실행 중 오류: {e}")
    print("\\n🔍 문제 해결 방법:")
    print("1. Ansys AEDT가 설치되어 있는지 확인")
    print("2. 올바른 버전이 설정되어 있는지 확인")
    print("3. 라이선스가 사용 가능한지 확인")

In [ ]:
# Maxwell GUI 표시 확인 및 추가 설정
try:
    print("\\n🖥️ Maxwell GUI 표시 설정을 확인합니다...")
    
    # Desktop이 GUI 모드로 실행되었는지 확인
    if hasattr(maxwell_desktop, 'odesktop'):
        print("✅ Maxwell Desktop GUI가 활성화되었습니다.")
        
        # Desktop을 전면으로 가져오기
        try:
            maxwell_desktop.odesktop.RestoreWindow()
            print("✅ Maxwell Desktop 창이 전면으로 이동되었습니다.")
        except:
            print("⚠️ Desktop 창 전면 이동 실패 (이미 표시 중일 수 있음)")
        
        # AEDT 메인 창 표시
        try:
            import win32gui
            import win32con
            
            def enum_windows_proc(hwnd, lParam):
                if win32gui.IsWindowVisible(hwnd):
                    window_text = win32gui.GetWindowText(hwnd)
                    if 'Ansys Electronics Desktop' in window_text or 'AEDT' in window_text:
                        win32gui.ShowWindow(hwnd, win32con.SW_RESTORE)
                        win32gui.SetForegroundWindow(hwnd)
                        print(f"✅ AEDT 창을 찾아 표시했습니다: {window_text}")
                        return False
                return True
            
            win32gui.EnumWindows(enum_windows_proc, 0)
            
        except ImportError:
            print("⚠️ win32gui를 사용할 수 없어 창 제어를 건너뜁니다.")
            print("💡 수동으로 작업 표시줄에서 AEDT 창을 클릭하여 활성화하세요.")
        except Exception as e:
            print(f"⚠️ 창 표시 중 오류: {e}")
    
    else:
        print("❌ Maxwell Desktop 객체에 문제가 있습니다.")
        
except Exception as e:
    print(f"❌ GUI 설정 확인 중 오류: {e}")

In [ ]:
# Maxwell 재시작 (GUI 강제 표시)
def restart_maxwell_with_gui():
    """Maxwell을 GUI 모드로 강제 재시작하는 함수"""
    try:
        print("🔄 Maxwell을 GUI 모드로 재시작합니다...")
        
        # 기존 Desktop 종료 (필요한 경우)
        try:
            if 'maxwell_desktop' in globals():
                maxwell_desktop.close()
                print("✅ 기존 Maxwell Desktop을 종료했습니다.")
        except:
            print("⚠️ 기존 Desktop 종료 중 오류 (무시)")
        
        # 새로운 Desktop을 GUI 모드로 시작
        from ansys.aedt.core import Desktop
        
        new_desktop = Desktop(
            specified_version="2025.1",
            new_desktop_session=True,
            non_graphical=False,      # GUI 모드
            close_on_exit=False,      # 종료 시 유지
            student_version=False     # 상용 버전 사용
        )
        
        print("✅ 새로운 Maxwell Desktop이 GUI 모드로 시작되었습니다.")
        
        # 새 프로젝트 생성
        from ansys.aedt.core import Maxwell2d
        
        project_name_gui = f"E8_Motor_GUI_{int(time.time())}"
        maxwell_2d_gui = Maxwell2d(
            project=project_name_gui,
            design="E8_Motor_GUI_Design",
            new_desktop=False,
            close_on_exit=False
        )
        
        print(f"✅ GUI 모드 Maxwell 2D 프로젝트 생성: {project_name_gui}")
        
        # 전역 변수 업데이트
        global maxwell_desktop, maxwell_2d, oDesktop, oProject, oDesign
        maxwell_desktop = new_desktop
        maxwell_2d = maxwell_2d_gui
        oDesktop = new_desktop.odesktop
        oProject = maxwell_2d_gui.oproject
        oDesign = maxwell_2d_gui.odesign
        
        return True
        
    except Exception as e:
        print(f"❌ Maxwell GUI 재시작 중 오류: {e}")
        return False

# GUI가 보이지 않는 경우 재시작 실행
print("\\n❓ Maxwell GUI가 보이지 않나요?")
print("아래 셀을 실행하여 GUI 모드로 재시작할 수 있습니다:")
print("restart_maxwell_with_gui()")

In [ ]:
# Maxwell GUI 재시작 실행
# 이 셀을 실행하여 Maxwell을 GUI 모드로 재시작하세요
restart_maxwell_with_gui()

In [ ]:
# Maxwell 3D 실행 옵션 (선택사항)
try:
    print("\\n🔧 Maxwell 3D 프로젝트도 생성하시겠습니까? (선택사항)")
    
    # Maxwell 3D 프로젝트 생성
    from ansys.aedt.core import Maxwell3d
    
    project_name_3d = f"E8_Motor_3D_Analysis_{int(time.time())}"
    maxwell_3d = Maxwell3d(
        project=project_name_3d,
        design="E8_Motor_3D_Design",
        new_desktop=False,  # 기존 desktop 사용
        close_on_exit=False
    )
    
    print(f"✅ Maxwell 3D 프로젝트가 생성되었습니다: {project_name_3d}")
    
    # 3D 프로젝트 정보
    print(f"\\n📊 Maxwell 3D 프로젝트 정보:")
    print(f"  프로젝트명: {maxwell_3d.project_name}")
    print(f"  디자인명: {maxwell_3d.design_name}")
    print(f"  솔루션 타입: {maxwell_3d.solution_type}")
    
except Exception as e:
    print(f"⚠️ Maxwell 3D 생성 중 오류: {e}")
    print("2D 프로젝트만 사용합니다.")

In [ ]:
# MotorCAD에서 생성된 Maxwell 스크립트 실행
try:
    print("\\n📜 MotorCAD에서 생성된 Maxwell 스크립트를 실행합니다...")
    
    # 앞서 생성된 Maxwell 스크립트 파일 찾기
    working_folder = os.getcwd()
    
    # 최근 생성된 Maxwell 스크립트 파일 찾기
    maxwell_scripts = []
    for file in os.listdir(working_folder):
        if 'maxwell' in file.lower() and file.endswith('.py'):
            maxwell_scripts.append(file)
    
    if maxwell_scripts:
        # 가장 최근 파일 선택
        latest_script = max(maxwell_scripts, key=lambda x: os.path.getctime(os.path.join(working_folder, x)))
        script_path = os.path.join(working_folder, latest_script)
        
        print(f"📁 발견된 Maxwell 스크립트: {latest_script}")
        
        # 스크립트 실행
        print("🔄 Maxwell 스크립트를 실행합니다...")
        
        # 방법 1: exec 사용하여 스크립트 실행
        try:
            with open(script_path, 'r', encoding='utf-8') as f:
                script_content = f.read()
            
            # 스크립트에서 필요한 변수들 설정
            exec(script_content, globals())
            print("✅ Maxwell 스크립트가 성공적으로 실행되었습니다!")
            
        except Exception as script_error:
            print(f"⚠️ 스크립트 직접 실행 실패: {script_error}")
            
            # 방법 2: Maxwell에서 스크립트 파일 실행
            try:
                maxwell_desktop.odesktop.RunScript(script_path)
                print("✅ Maxwell Desktop을 통해 스크립트가 실행되었습니다!")
            except Exception as desktop_error:
                print(f"❌ Maxwell Desktop 스크립트 실행 실패: {desktop_error}")
    
    else:
        print("❌ Maxwell 스크립트 파일을 찾을 수 없습니다.")
        print("MotorCAD에서 Maxwell로 내보내기를 먼저 실행해주세요.")
        
        # 사용 가능한 .py 파일들 표시
        py_files = [f for f in os.listdir(working_folder) if f.endswith('.py')]
        if py_files:
            print("\\n📄 사용 가능한 Python 파일들:")
            for py_file in py_files:
                print(f"  - {py_file}")
                
except Exception as e:
    print(f"❌ Maxwell 스크립트 실행 중 오류: {e}")

print("\\n🎉 Ansys Maxwell이 성공적으로 실행되었습니다!")
print("\\n📋 다음 단계:")
print("1. Maxwell GUI에서 모델 확인")
print("2. 메쉬 설정 및 경계 조건 확인") 
print("3. 해석 설정 및 실행")
print("4. 결과 확인 및 후처리")